In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

import src.features as F
import src.split as S
import src.metrics as M

df = pd.read_parquet("../data/processed/clean.parquet")
feats = F.build_features(df)
train, val, _ = S.time_split(feats)         # test is deliberately not bound to a name

y_tr, y_val = train["duration"], val["duration"]

# 1. constant: train median
pred_const = np.full(len(val), y_tr.median())

# 2. raw upstream estimate; NaN -> train median of the estimate
est_fill = train["estimated_total"].median()
pred_raw = val["estimated_total"].fillna(est_fill)

# 3. calibrated: duration ~ a + b * estimated_total, fit on train only
lin = LinearRegression().fit(train[["estimated_total"]].fillna(est_fill), y_tr)
pred_cal = lin.predict(val[["estimated_total"]].fillna(est_fill))

results = pd.DataFrame({
    "constant (train median)": M.evaluate(y_val, pred_const),
    "raw estimate": M.evaluate(y_val, pred_raw),
    "calibrated estimate": M.evaluate(y_val, pred_cal),
}).T.round(3)
display(results)
print(f"calibrated: duration = {lin.intercept_:.0f} + {lin.coef_[0]:.2f} * estimated_total")

,mae_min,rmse_min,within_10min,bias_min,over_pct
constant (train median),12.611,17.154,0.505,-3.190,0.483
raw estimate,32.916,36.674,0.026,-32.916,0.000
calibrated estimate,12.248,16.153,0.493,0.197,0.578


calibrated: duration = 1794 + 1.23 * estimated_total
